# model_experiment_ARIMA_SARIMA

Classical statistical models — deliberately compute-light, theory-heavy (per the assignment: distinguish them theoretically, don't over-train).

## Theory

**ARIMA(p, d, q)** models a single series as a combination of:
- **AR(p)** — autoregression: today's value depends linearly on the last *p* values;
- **I(d)** — integration: difference the series *d* times first, because AR/MA theory requires **stationarity** (constant mean/variance over time);
- **MA(q)** — moving average: today's value depends on the last *q* forecast errors.

Stationarity is checked with the **ADF test** (H0: unit root / non-stationary); p and q are read from the **PACF** and **ACF** plots respectively.

**SARIMA(p,d,q)(P,D,Q,s)** adds the same three components *at seasonal lag s* — for weekly data with yearly seasonality, s=52. Plain ARIMA is structurally blind to a 52-week cycle: an AR(p) term with small p cannot reach 52 weeks back.

**Why classical models can't win this competition:** they are per-series (3331 separate fits, no cross-series learning), fail on short/cold-start series, can't use exogenous features easily at this scale, and s=52 fits are slow. This is exactly the motivation for global models (LightGBM/DLinear/PatchTST). We demonstrate: (1) the full ARIMA workflow on one large series; (2) a hybrid SARIMA-top-50 + seasonal-naive submission for an honest score.

MLflow experiment: **ARIMA_SARIMA_Training**. CPU only, ~10 min.

In [ ]:
# Kaggle bootstrap — does nothing when running locally.
import os
ON_KAGGLE = os.path.exists("/kaggle")
if ON_KAGGLE:
    os.system("pip install -q mlflow")
    if not os.path.exists("walmart-sales-forecasting"):
        os.system("git clone https://github.com/ekatsirekidze/walmart-sales-forecasting.git")
    import sys; sys.path.insert(0, "/kaggle/working/walmart-sales-forecasting")
    import glob, shutil, zipfile
    src = glob.glob("/kaggle/input/*walmart*") + glob.glob("/kaggle/input/*/*walmart*")
    assert src, "competition data not attached"
    os.makedirs("data", exist_ok=True)
    for f in glob.glob(src[0] + "/*"):
        (zipfile.ZipFile(f).extractall("data") if f.endswith(".zip") else shutil.copy(f, "data"))
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ["MLFLOW_TRACKING_URI"] = s.get_secret("MLFLOW_TRACKING_URI")
    os.environ["MLFLOW_TRACKING_USERNAME"] = s.get_secret("MLFLOW_TRACKING_USERNAME")
    os.environ["MLFLOW_TRACKING_PASSWORD"] = s.get_secret("MLFLOW_TRACKING_PASSWORD")

In [ ]:
import sys, pathlib, warnings
sys.path.insert(0, str(pathlib.Path().resolve()))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import mlflow
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from src.data import load_raw, make_submission
from src.metrics import wmae_report
from src.validation import split_fold
from src.mlflow_utils import setup_mlflow
from src.postprocess import naive_lag52, apply_christmas_shift

train, test, features, stores = load_raw("data" if ON_KAGGLE else None)
setup_mlflow("ARIMA_SARIMA_Training")

## 1. Stationarity: ADF test + ACF/PACF (demo series: Store 20 / Dept 92)

In [ ]:
tr, va = split_fold(train, 1)
s = tr[(tr.Store == 20) & (tr.Dept == 92)].set_index("Date")["Weekly_Sales"].asfreq("W-FRI")
sv = va[(va.Store == 20) & (va.Dept == 92)].set_index("Date")["Weekly_Sales"].asfreq("W-FRI")

adf_raw = adfuller(s.dropna())
adf_diff = adfuller(s.diff().dropna())
print(f"ADF raw series:    p = {adf_raw[1]:.4f}  -> non-stationary (can't reject unit root)")
print(f"ADF 1st difference: p = {adf_diff[1]:.6f} -> stationary => d=1")

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
axes[0, 0].plot(s); axes[0, 0].set_title("raw series")
axes[0, 1].plot(s.diff()); axes[0, 1].set_title("1st difference")
plot_acf(s.diff().dropna(), lags=60, ax=axes[1, 0])
plot_pacf(s.diff().dropna(), lags=40, ax=axes[1, 1])
plt.tight_layout(); plt.show()
# note the ACF spike near lag 52 -> yearly seasonality that plain ARIMA cannot model

## 2. ARIMA vs SARIMA on the demo series

ARIMA(1,1,1) has no mechanism to reach 52 weeks back; SARIMA(1,1,1)(0,1,1,52) adds seasonal differencing + a seasonal MA term.

In [ ]:
arima = SARIMAX(s, order=(1, 1, 1)).fit(disp=False)
fc_arima = arima.forecast(39)

sarima = SARIMAX(s, order=(1, 1, 1), seasonal_order=(0, 1, 1, 52)).fit(disp=False)
fc_sarima = sarima.forecast(39)

mae_arima = np.mean(np.abs(fc_arima.values - sv.values))
mae_sarima = np.mean(np.abs(fc_sarima.values - sv.values))

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(s[-70:], label="history")
ax.plot(sv, label="actual", color="black")
ax.plot(fc_arima, label=f"ARIMA (MAE {mae_arima:,.0f})", ls="--")
ax.plot(fc_sarima, label=f"SARIMA (MAE {mae_sarima:,.0f})", ls="--")
ax.legend(); ax.set_title("Store 20 / Dept 92 — ARIMA is flat, SARIMA follows the seasonal shape")
plt.show()

with mlflow.start_run(run_name="ARIMA_vs_SARIMA_Demo"):
    mlflow.log_params({"series": "store20_dept92", "arima": "(1,1,1)",
                       "sarima": "(1,1,1)(0,1,1,52)",
                       "adf_raw_p": round(adf_raw[1], 4), "adf_diff_p": round(adf_diff[1], 6)})
    mlflow.log_metrics({"arima_val_mae": mae_arima, "sarima_val_mae": mae_sarima})
print(f"ARIMA {mae_arima:,.0f} vs SARIMA {mae_sarima:,.0f}")

## 3. Hybrid at scale: SARIMA on the top-50 series, seasonal naive for the rest

Fitting SARIMA(s=52) on all 3331 series would take hours and fail on short series — the scaling wall that motivates global models. Top-50 series by volume carry a large share of total sales, so this hybrid is a fair demonstration of what the classical approach can achieve.

In [ ]:
def sarima_hybrid(history, target, n_series=50):
    pred = pd.Series(naive_lag52(history, target), index=target.index)
    dep_med = history.groupby("Dept")["Weekly_Sales"].median()
    pred = pred.fillna(target["Dept"].map(dep_med).set_axis(target.index)).fillna(0)

    top = history.groupby(["Store", "Dept"])["Weekly_Sales"].sum().nlargest(n_series).index
    n_ok = 0
    horizon_dates = pd.DatetimeIndex(sorted(target["Date"].unique()))
    for st, dp in top:
        y = (history[(history.Store == st) & (history.Dept == dp)]
             .set_index("Date")["Weekly_Sales"].asfreq("W-FRI").fillna(0))
        try:
            m = SARIMAX(y, order=(1, 1, 1), seasonal_order=(0, 1, 1, 52)).fit(disp=False)
            fc = m.forecast(len(horizon_dates))
            fc.index = horizon_dates
            rows = target[(target.Store == st) & (target.Dept == dp)]
            pred.loc[rows.index] = rows["Date"].map(fc).to_numpy()
            n_ok += 1
        except Exception:
            pass  # keep the naive prediction for this series
    return pred, n_ok

pred_va, n_ok = sarima_hybrid(tr, va)
rep = wmae_report(va["Weekly_Sales"], pred_va, va["IsHoliday"])
with mlflow.start_run(run_name="SARIMA_CV_Fold1_Hybrid"):
    mlflow.log_params({"n_sarima_series": 50, "order": "(1,1,1)",
                       "seasonal_order": "(0,1,1,52)", "rest": "seasonal_naive", "fold": 1})
    mlflow.log_metrics({**rep, "sarima_fits_ok": n_ok})
print(f"fold 1 hybrid ({n_ok} SARIMA fits): {rep}")

## 4. Final: hybrid on full train -> test submission

In [ ]:
pred_test, n_ok = sarima_hybrid(train, test)
sub = test[["Store", "Dept", "Date"]].copy()
sub["pred"] = pred_test.to_numpy()
sub = apply_christmas_shift(sub, pred_col="pred")

with mlflow.start_run(run_name="SARIMA_Final"):
    mlflow.log_params({"n_sarima_series": 50, "order": "(1,1,1)",
                       "seasonal_order": "(0,1,1,52)", "rest": "seasonal_naive",
                       "post": "christmas_shift"})
    mlflow.log_metrics({"fold1_wmae": rep["wmae"], "sarima_fits_ok": n_ok})

make_submission(sub, "pred", "submission_sarima.csv")
print(f"wrote submission_sarima.csv ({n_ok} SARIMA fits)")

## Conclusions (for the README)

1. ARIMA vs SARIMA on one series makes the seasonal term's value visible: the ARIMA forecast is essentially flat while SARIMA tracks the yearly shape.
2. The classical workflow (ADF → difference → ACF/PACF → fit) is per-series manual labor; on 3331 series it becomes a compute + robustness wall.
3. The hybrid score lands near the seasonal-naive baseline — SARIMA's gains on 50 big series barely move a metric dominated by thousands of series.
4. These limitations are the historical reason the field moved to global ML models — which is the story of every other notebook in this repo.